# Tarea 2 — Parte C: las seis preguntas

Este notebook usa **las mismas funciones** que `python -m t2.preguntas` (en `t2/preguntas.py`), así que
script y notebook no pueden divergir. Requiere el pipeline ya corrido (`./run.sh --sin-experimentos`) y el kernel
del `.venv` (`.venv/bin/python -m pip install ipykernel`).

Cada medición: 1 corrida de calentamiento (descartada) + **mediana de 5**.

In [ ]:
import json, pandas as pd
from t2 import preguntas, redis_util, hbase_util
from t2.medicion import hardware
r, conn = redis_util.conectar(), hbase_util.conectar()
hardware()

## P1 · Ahora — Redis contra HBase

In [ ]:
p1 = preguntas.p1(r, conn)
display(pd.DataFrame(p1["ultimos"]))
print("top 5 Redis:", p1["top5_redis"]); print("top 5 HBase:", p1["top5_hbase"])
print("tiempos:", json.dumps(p1["tiempos"], indent=1), "\nHBase es", p1["veces_mas_lento_hbase"], "veces más lento")
pd.json_normalize(p1["medicion_vs_llegada"])

## P2 · Ventana de 10 minutos — solo Redis

In [ ]:
p2 = preguntas.p2(r)
print("ancla:", p2["ancla"], "| tiempo:", p2["tiempo"])
display(pd.DataFrame(p2["promedios_10min"]).T)
display(pd.DataFrame(p2["memoria"]))
p2["extrapolacion_sin_recorte"]

## P3 · Un rango — HBase (scan por rango vs tabla completa + filtro)

In [ ]:
p3 = preguntas.p3(conn)
print(p3["lecturas"], "lecturas; tiempos:", p3["tiempos"], "; veces más lento:", p3["veces_mas_lento"])
pd.Series(p3["promedio_por_cuarto_de_hora"], name="promedio flujo")

## P4 · Dos patrones de acceso — catálogo Redis + `lecturas` vs `lecturas_tipo`

In [ ]:
p4 = preguntas.p4(r, conn)
display(pd.DataFrame(p4["resultado"]).T)
print(json.dumps(p4["tiempos"], indent=1)); p4["costo_segunda_tabla"]

## P5 · Hotspotting y salting (escribe 3 tablas y espera ~20 s a JMX)

In [ ]:
p5 = preguntas.p5(conn)
display(pd.DataFrame(p5["tabla_3xregiones_jmx"]).T)
display(pd.DataFrame(p5["tabla_3xregiones_filas"]).T)
print("salt por estación:", p5["salt_por_estacion"])
pd.DataFrame(p5["tiempos_lectura"]).T

## P6 · La propia — ¿qué tan raro es el ahora? (Redis = qué, HBase = cuándo)

In [ ]:
p6 = preguntas.p6(r, conn)
print(p6["pregunta"]); print("tiempo:", p6["tiempo"])
pd.DataFrame(p6["resultado"])